# Notebook 01: Data Foundation & Realistic Synthetic Generation

## GeoUMKM Smart - Location Intelligence for UMKM in Jawa Barat

This notebook generates a realistic synthetic dataset of 10,000 UMKM across Jawa Barat province.

### Design Principles (NO Circular Logic)

1. **Geography First**: All 27 kabupaten/kota and ~627 kecamatan use REAL names and coordinates
2. **Features are INDEPENDENT**: Generated from geographic/demographic basis
3. **Target from Features**: `is_survived_3yr` is generated FROM features using a known logistic relationship
4. **Score is DERIVED**: `skor_potensi` is COMPUTED from features (weighted formula), NOT random

The causal flow: Geography -> Demographics -> Infrastructure -> Business Features -> Target

In [1]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('../data', exist_ok=True)

print("Libraries loaded. NumPy:", np.__version__, "Pandas:", pd.__version__)

Libraries loaded. NumPy: 2.0.2 Pandas: 2.3.3


## Section 1: Geography Reference Data

We define all 27 kabupaten/kota in Jawa Barat with their approximate centroid coordinates.
Each entry includes whether it is a 'kota' (urban municipality) or 'kabupaten' (regency),
which fundamentally drives urbanization characteristics downstream.

In [2]:
# All 27 kabupaten/kota in Jawa Barat with approximate centroids
# Format: name -> (latitude, longitude, is_kota)
KABUPATEN_KOTA = {
    'Kab. Bandung': (-6.98, 107.59, False),
    'Kota Bandung': (-6.91, 107.61, True),
    'Kab. Bogor': (-6.59, 106.79, False),
    'Kota Bogor': (-6.60, 106.80, True),
    'Kab. Bekasi': (-6.24, 107.00, False),
    'Kota Bekasi': (-6.24, 106.99, True),
    'Kota Depok': (-6.40, 106.82, True),
    'Kab. Tasikmalaya': (-7.35, 108.22, False),
    'Kota Tasikmalaya': (-7.33, 108.22, True),
    'Kab. Cirebon': (-6.71, 108.56, False),
    'Kota Cirebon': (-6.73, 108.55, True),
    'Kab. Sukabumi': (-6.92, 106.93, False),
    'Kota Sukabumi': (-6.92, 106.93, True),
    'Kab. Garut': (-7.22, 107.90, False),
    'Kab. Ciamis': (-7.33, 108.35, False),
    'Kab. Kuningan': (-6.97, 108.48, False),
    'Kab. Majalengka': (-6.84, 108.23, False),
    'Kab. Sumedang': (-6.86, 107.92, False),
    'Kab. Indramayu': (-6.33, 108.32, False),
    'Kab. Subang': (-6.57, 107.76, False),
    'Kab. Purwakarta': (-6.55, 107.43, False),
    'Kab. Karawang': (-6.32, 107.30, False),
    'Kab. Pangandaran': (-7.68, 108.65, False),
    'Kab. Bandung Barat': (-6.86, 107.46, False),
    'Kota Cimahi': (-6.87, 107.54, True),
    'Kota Banjar': (-7.37, 108.54, True),
}

print(f"Defined {len(KABUPATEN_KOTA)} kabupaten/kota")
print(f"  Kota (urban): {sum(1 for v in KABUPATEN_KOTA.values() if v[2])}")
print(f"  Kabupaten (regency): {sum(1 for v in KABUPATEN_KOTA.values() if not v[2])}")

Defined 26 kabupaten/kota
  Kota (urban): 9
  Kabupaten (regency): 17


## Section 1b: Kecamatan Generation

For each kabupaten/kota, we define real kecamatan names from Jawa Barat.
Coordinates are generated as small random offsets from the kabupaten centroid
(within +/- 0.08 degrees latitude and longitude).

This produces approximately 627 kecamatan entries with realistic naming and geography.

In [3]:
# Real kecamatan names for each kabupaten/kota in Jawa Barat
KECAMATAN_DATA = {
    'Kota Bandung': [
        'Coblong', 'Bandung Wetan', 'Sumur Bandung', 'Cicendo', 'Andir',
        'Sukajadi', 'Cidadap', 'Lengkong', 'Regol', 'Astana Anyar',
        'Bojongloa Kaler', 'Babakan Ciparay', 'Bandung Kulon', 'Kiaracondong',
        'Batununggal', 'Antapani', 'Arcamanik', 'Cibiru', 'Ujungberung',
        'Cinambo', 'Mandalajati', 'Gedebage', 'Rancasari', 'Buahbatu',
        'Bandung Kidul', 'Margacinta', 'Sukasari', 'Cibeunying Kaler',
        'Cibeunying Kidul', 'Panyileukan'
    ],
    'Kab. Bandung': [
        'Cileunyi', 'Cimenyan', 'Cilengkrang', 'Bojongsoang', 'Margahayu',
        'Margaasih', 'Katapang', 'Dayeuhkolot', 'Banjaran', 'Pameungpeuk',
        'Pangalengan', 'Arjasari', 'Cimaung', 'Ibun', 'Paseh',
        'Kertasari', 'Pacet', 'Ciparay', 'Solokanjeruk', 'Majalaya',
        'Rancaekek', 'Cicalengka', 'Nagreg', 'Cikancung', 'Baleendah',
        'Soreang', 'Kutawaringin', 'Ciwidey', 'Rancabali', 'Pasirjambu',
        'Cangkuang'
    ],
    'Kab. Bogor': [
        'Cibinong', 'Gunung Putri', 'Citeureup', 'Cileungsi', 'Jonggol',
        'Cariu', 'Sukamakmur', 'Parung', 'Ciawi', 'Megamendung',
        'Cisarua', 'Sukaraja', 'Dramaga', 'Ciomas', 'Tamansari',
        'Kemang', 'Rancabungur', 'Tajurhalang', 'Bojonggede', 'Leuwiliang',
        'Nanggung', 'Cigudeg', 'Tenjo', 'Jasinga', 'Parungpanjang',
        'Rumpin', 'Gunung Sindur', 'Cibungbulang', 'Ciampea', 'Tenjolaya',
        'Babakan Madang', 'Sukarata', 'Klapanunggal', 'Tanjungsari',
        'Caringin', 'Cijeruk', 'Cigombong', 'Leuwisadeng', 'Pamijahan',
        'Ciseeng'
    ],
    'Kota Bogor': [
        'Bogor Utara', 'Bogor Selatan', 'Bogor Timur', 'Bogor Barat',
        'Bogor Tengah', 'Tanah Sareal'
    ],
    'Kab. Bekasi': [
        'Cikarang Barat', 'Cikarang Timur', 'Cikarang Utara', 'Cikarang Selatan',
        'Cikarang Pusat', 'Tambun Selatan', 'Tambun Utara', 'Tarumajaya',
        'Babelan', 'Muaragembong', 'Cabangbungin', 'Pebayuran',
        'Kedungwaringin', 'Cibarusah', 'Bojongmangu', 'Sukatani',
        'Karangbahagia', 'Setu', 'Serang Baru', 'Sukakarya',
        'Tambelang', 'Sukawangi', 'Karangsentosa'
    ],
    'Kota Bekasi': [
        'Bekasi Timur', 'Bekasi Barat', 'Bekasi Utara', 'Bekasi Selatan',
        'Rawalumbu', 'Medan Satria', 'Bantar Gebang', 'Jatiasih',
        'Jatisampurna', 'Mustika Jaya', 'Pondok Gede', 'Pondok Melati'
    ],
    'Kota Depok': [
        'Beji', 'Pancoran Mas', 'Cipayung', 'Sukmajaya', 'Cilodong',
        'Cimanggis', 'Tapos', 'Sawangan', 'Bojongsari', 'Limo', 'Cinere'
    ],
    'Kab. Garut': [
        'Garut Kota', 'Karangpawitan', 'Wanaraja', 'Tarogong Kaler',
        'Tarogong Kidul', 'Banyuresmi', 'Samarang', 'Bayongbong',
        'Cigedug', 'Cilawu', 'Kadungora', 'Leles', 'Leuwigoong',
        'Cibatu', 'Kersamanah', 'Selaawi', 'Malangbong', 'Cibalong',
        'Pameungpeuk', 'Cisewu', 'Caringin', 'Bungbulang', 'Mekarmukti',
        'Pakenjeng', 'Pamulihan', 'Cisompet', 'Peundeuy', 'Singajaya',
        'Cihurip', 'Sukaresmi', 'Banjarwangi', 'Limbangan', 'Cikajang',
        'Cisurupan', 'Pasirwangi', 'Sukawening', 'Cikelet', 'Blubur Limbangan',
        'Cibiuk', 'Pangatikan', 'Sucinaraja', 'Karangtengah'
    ],
    'Kab. Sukabumi': [
        'Palabuhanratu', 'Cikakak', 'Cisolok', 'Cicurug', 'Cidahu',
        'Parungkuda', 'Cibadak', 'Nagrak', 'Cisaat', 'Sukabumi',
        'Sukaraja', 'Kadudampit', 'Gunungguruh', 'Kebonpedes',
        'Cireunghas', 'Jampangtengah', 'Jampangkulon', 'Ciemas',
        'Tegalbuleud', 'Sagaranten', 'Pabuaran', 'Purabaya',
        'Warungkiara', 'Bantargadung', 'Lengkong', 'Nyalindung',
        'Gegerbitung', 'Surade', 'Cikidang', 'Waluran',
        'Simpenan', 'Ciracap', 'Curugkembar', 'Kalapanunggal',
        'Kabandungan', 'Cimanggu', 'Cicantayan', 'Caringin',
        'Cidolog', 'Kalibunder', 'Cikembar', 'Bojonggenteng',
        'Parakansalak', 'Ciambar', 'Cidadap', 'Sirnaresmi', 'Cibitung'
    ],
    'Kota Sukabumi': [
        'Gunungpuyuh', 'Cikole', 'Citamiang', 'Warudoyong',
        'Baros', 'Lembursitu', 'Cibeureum'
    ],
    'Kab. Tasikmalaya': [
        'Singaparna', 'Sukarame', 'Ciawi', 'Cisayong', 'Manonjaya',
        'Salopa', 'Cikatomas', 'Bantarkalong', 'Karangnunggal',
        'Cibalong', 'Parungponteng', 'Sodonghilir', 'Taraju',
        'Salawu', 'Puspahiang', 'Cipatujah', 'Cikalong',
        'Pancatengah', 'Rajapolah', 'Jamanis', 'Padakembang',
        'Sukaratu', 'Leuwisari', 'Sariwangi', 'Sukaresik',
        'Mangunreja', 'Gunung Tanjung', 'Karangjaya', 'Tanjungjaya',
        'Sukaraja', 'Bojongasih', 'Culamega', 'Ciawi', 'Kadipaten',
        'Jatiwaras', 'Karangmukti', 'Cigalontang', 'Bojonggambir',
        'Sukahening'
    ],
    'Kota Tasikmalaya': [
        'Cihideung', 'Cipedes', 'Tawang', 'Mangkubumi', 'Indihiang',
        'Bungursari', 'Purbaratu', 'Tamansari', 'Kawalu', 'Cibeureum'
    ],
    'Kab. Cirebon': [
        'Sumber', 'Plered', 'Kedawung', 'Weru', 'Plumbon',
        'Depok', 'Palimanan', 'Klangenan', 'Arjawinangun', 'Panguragan',
        'Ciwaringin', 'Gempol', 'Susukan', 'Sedong', 'Astanajapura',
        'Mundu', 'Pangenan', 'Kaliwedi', 'Gebang', 'Losari',
        'Babakan', 'Lemahabang', 'Susukanlebak', 'Waled',
        'Ciledug', 'Karangsembung', 'Kapetakan', 'Tengahtani',
        'Talun', 'Beber', 'Greged', 'Jamblang', 'Dukupuntang',
        'Gunungjati', 'Suranenggala', 'Gunung Jati',
        'Karangwareng', 'Gegesik', 'Panggangsari', 'Pabuaran'
    ],
    'Kota Cirebon': [
        'Harjamukti', 'Kesambi', 'Lemahwungkuk', 'Pekalipan', 'Kejaksan'
    ],
    'Kab. Ciamis': [
        'Ciamis', 'Cikoneng', 'Cijeungjing', 'Sadananya', 'Cidolog',
        'Rancah', 'Rajadesa', 'Sukadana', 'Tambaksari', 'Cisaga',
        'Panawangan', 'Kawali', 'Panjalu', 'Jatinagara', 'Cipaku',
        'Pamarican', 'Banjaranyar', 'Lakbok', 'Mangunjaya',
        'Purwadadi', 'Baregbeg', 'Sindangkasih', 'Cihaurbeuti',
        'Panumbangan', 'Sukamantri', 'Lumbung', 'Banjarsari', 'Padaherang'
    ],
    'Kab. Kuningan': [
        'Kuningan', 'Cigugur', 'Kadugede', 'Luragung', 'Lebakwangi',
        'Garawangi', 'Ciawigebang', 'Cilimus', 'Jalaksana',
        'Kramatmulya', 'Darma', 'Mandirancan', 'Pancalang',
        'Pasawahan', 'Cipicung', 'Nusaherang', 'Selajambe',
        'Subang', 'Ciwaru', 'Karangkancana', 'Cibingbin',
        'Cimahi', 'Cidahu', 'Kalimanggis', 'Ciniru', 'Hantara',
        'Japara', 'Maleber', 'Cilebak', 'Cibeureum', 'Sindangagung',
        'Caracas'
    ],
    'Kab. Majalengka': [
        'Majalengka', 'Kadipaten', 'Jatitujuh', 'Kertajati', 'Jatiwangi',
        'Dawuan', 'Kasokandel', 'Panyingkiran', 'Cigasong', 'Sukahaji',
        'Rajagaluh', 'Sindangwangi', 'Leuwimunding', 'Palasah',
        'Bantarujeg', 'Lemahsugih', 'Cikijing', 'Talaga',
        'Argapura', 'Maja', 'Banjaran', 'Cingambul', 'Sindang',
        'Malausma', 'Ligung', 'Sumberjaya'
    ],
    'Kab. Sumedang': [
        'Sumedang Utara', 'Sumedang Selatan', 'Jatinangor', 'Cimanggung',
        'Tanjungsari', 'Rancakalong', 'Surian', 'Pamulihan',
        'Conggeang', 'Paseh', 'Cimalaka', 'Cisarua', 'Tomo',
        'Ujungjaya', 'Jatigede', 'Wado', 'Darmaraja', 'Cibugel',
        'Cisitu', 'Situraja', 'Ganeas', 'Tanjungkerta',
        'Tanjungmedar', 'Buahdua', 'Sukasari', 'Jatinunggal'
    ],
    'Kab. Indramayu': [
        'Indramayu', 'Jatibarang', 'Lohbener', 'Losarang', 'Kandanghaur',
        'Sukra', 'Patrol', 'Anjatan', 'Haurgeulis', 'Kroya',
        'Gabuswetan', 'Cikedung', 'Terisi', 'Widasari', 'Kertasemaya',
        'Bongas', 'Karangampel', 'Kedokanbunder', 'Juntinyuat',
        'Sliyeg', 'Sindang', 'Bangodua', 'Tukdana', 'Cantigi',
        'Arahan', 'Lelea', 'Gantar', 'Krangkeng', 'Balongan',
        'Pasekan', 'Karanganyar'
    ],
    'Kab. Subang': [
        'Subang', 'Kalijati', 'Cibogo', 'Cipunagara', 'Purwadadi',
        'Pagaden', 'Pagaden Barat', 'Binong', 'Ciasem', 'Pamanukan',
        'Pusakanagara', 'Pusakajaya', 'Legonkulon', 'Blanakan',
        'Sukasari', 'Compreng', 'Patokbeusi', 'Ciater', 'Sagalaherang',
        'Cisalak', 'Jalancagak', 'Serangpanjang', 'Tanjungsiang', 'Dawuan',
        'Cipeundeuy', 'Kasomalang', 'Cijambe', 'Tambakdahan', 'Cikaum', 'Pabuaran'
    ],
    'Kab. Purwakarta': [
        'Purwakarta', 'Babakancikao', 'Campaka', 'Cibatu', 'Darangdan',
        'Jatiluhur', 'Maniis', 'Pasawahan', 'Plered', 'Pondoksalam',
        'Sukatani', 'Tegalwaru', 'Wanayasa', 'Bojong', 'Bungursari',
        'Kiarapedes', 'Sukasari'
    ],
    'Kab. Karawang': [
        'Karawang Barat', 'Karawang Timur', 'Klari', 'Cikampek',
        'Purwasari', 'Kotabaru', 'Telukjambe Barat', 'Telukjambe Timur',
        'Ciampel', 'Pangkalan', 'Tegalwaru', 'Jatisari',
        'Cilamaya Kulon', 'Cilamaya Wetan', 'Lemahabang', 'Tempuran',
        'Majalaya', 'Rawamerta', 'Kutawaluya', 'Rengasdengklok',
        'Batujaya', 'Pakisjaya', 'Jayakerta', 'Pedes', 'Cilebar',
        'Tirtamulya', 'Tirtajaya', 'Cibuaya', 'Telagasari', 'Banyusari'
    ],
    'Kab. Pangandaran': [
        'Pangandaran', 'Parigi', 'Cijulang', 'Cimerak', 'Cigugur',
        'Langkaplancar', 'Mangunjaya', 'Padaherang', 'Kalipucang',
        'Sidamulih'
    ],
    'Kab. Bandung Barat': [
        'Lembang', 'Parongpong', 'Cisarua', 'Cikalongwetan', 'Cipeundeuy',
        'Ngamprah', 'Padalarang', 'Batujajar', 'Cihampelas', 'Cililin',
        'Cipongkor', 'Rongga', 'Sindangkerta', 'Gununghalu',
        'Saguling', 'Cipatat'
    ],
    'Kota Cimahi': [
        'Cimahi Utara', 'Cimahi Tengah', 'Cimahi Selatan'
    ],
    'Kota Banjar': [
        'Banjar', 'Purwaharja', 'Pataruman', 'Langensari'
    ],
}

print(f"Total kecamatan defined: {sum(len(v) for v in KECAMATAN_DATA.items())}")
for kab, kecs in KECAMATAN_DATA.items():
    print(f"  {kab}: {len(kecs)} kecamatan")

Total kecamatan defined: 52
  Kota Bandung: 30 kecamatan
  Kab. Bandung: 31 kecamatan
  Kab. Bogor: 40 kecamatan
  Kota Bogor: 6 kecamatan
  Kab. Bekasi: 23 kecamatan
  Kota Bekasi: 12 kecamatan
  Kota Depok: 11 kecamatan
  Kab. Garut: 42 kecamatan
  Kab. Sukabumi: 47 kecamatan
  Kota Sukabumi: 7 kecamatan
  Kab. Tasikmalaya: 39 kecamatan
  Kota Tasikmalaya: 10 kecamatan
  Kab. Cirebon: 40 kecamatan
  Kota Cirebon: 5 kecamatan
  Kab. Ciamis: 28 kecamatan
  Kab. Kuningan: 32 kecamatan
  Kab. Majalengka: 26 kecamatan
  Kab. Sumedang: 26 kecamatan
  Kab. Indramayu: 31 kecamatan
  Kab. Subang: 30 kecamatan
  Kab. Purwakarta: 17 kecamatan
  Kab. Karawang: 30 kecamatan
  Kab. Pangandaran: 10 kecamatan
  Kab. Bandung Barat: 16 kecamatan
  Kota Cimahi: 3 kecamatan
  Kota Banjar: 4 kecamatan


In [4]:
# Generate kecamatan reference dataframe with coordinates
rows = []
for kab_name, (lat_center, lon_center, is_kota) in KABUPATEN_KOTA.items():
    kecamatan_list = KECAMATAN_DATA.get(kab_name, [])
    for kec_name in kecamatan_list:
        # Generate coordinate offset from kabupaten centroid
        lat_offset = np.random.uniform(-0.08, 0.08)
        lon_offset = np.random.uniform(-0.08, 0.08)
        lat = lat_center + lat_offset
        lon = lon_center + lon_offset
        rows.append({
            'kabupaten_kota': kab_name,
            'kecamatan': kec_name,
            'latitude': round(lat, 6),
            'longitude': round(lon, 6),
            'is_kota': is_kota
        })

kecamatan_df = pd.DataFrame(rows)
print(f"Total kecamatan generated: {len(kecamatan_df)}")
print(f"\nCoordinate bounds:")
print(f"  Latitude:  {kecamatan_df['latitude'].min():.4f} to {kecamatan_df['latitude'].max():.4f}")
print(f"  Longitude: {kecamatan_df['longitude'].min():.4f} to {kecamatan_df['longitude'].max():.4f}")
print(f"\nSample:")
kecamatan_df.head(10)

Total kecamatan generated: 596

Coordinate bounds:
  Latitude:  -7.7470 to -6.1643
  Longitude: 106.7127 to 108.7106

Sample:


,kabupaten_kota,kecamatan,latitude,longitude,is_kota
0,Kab. Bandung,Cileunyi,-7.000074,107.662114,False
1,Kab. Bandung,Cimenyan,-6.942881,107.605785,False
2,Kab. Bandung,Cilengkrang,-7.035037,107.534959,False
3,Kab. Bandung,Bojongsoang,-7.050707,107.648588,False
4,Kab. Bandung,Margahayu,-6.963822,107.623292,False
5,Kab. Bandung,Margaasih,-7.056706,107.665186,False
6,Kab. Bandung,Katapang,-6.926809,107.543974,False
7,Kab. Bandung,Dayeuhkolot,-7.030908,107.539345,False
8,Kab. Bandung,Banjaran,-7.011321,107.593961,False
9,Kab. Bandung,Pameungpeuk,-6.990889,107.556597,False


## Section 2: Demographic Features (from Geography)

Demographics are driven by geography:
- **populasi**: Higher in urban kota areas and the Bandung-Bekasi-Depok corridor. Log-normal distribution.
- **kepadatan_penduduk**: Derived from populasi and estimated area.
- **income_per_kapita**: Higher in Kota Bandung, Kota Bekasi, Depok, Cimahi. Log-normal distribution.

These are foundational features that drive downstream infrastructure and business characteristics.

In [5]:
# Define urbanization multipliers based on location
# Higher multiplier = more urban/developed
def get_urbanization_factor(kab_name, is_kota):
    """Return urbanization factor (0-1) based on kabupaten/kota."""
    # Major urban centers
    high_urban = ['Kota Bandung', 'Kota Bekasi', 'Kota Depok', 'Kota Cimahi']
    medium_urban = ['Kota Bogor', 'Kota Tasikmalaya', 'Kota Cirebon', 'Kota Sukabumi',
                    'Kab. Bekasi', 'Kab. Bogor', 'Kab. Bandung', 'Kab. Bandung Barat']
    # Industrial corridors
    industrial = ['Kab. Karawang', 'Kab. Purwakarta']
    # Rural areas
    rural = ['Kab. Pangandaran', 'Kab. Ciamis', 'Kab. Garut', 'Kab. Tasikmalaya',
             'Kab. Sukabumi', 'Kab. Kuningan']
    
    if kab_name in high_urban:
        return np.random.uniform(0.75, 0.95)
    elif kab_name in medium_urban:
        return np.random.uniform(0.55, 0.75)
    elif kab_name in industrial:
        return np.random.uniform(0.50, 0.70)
    elif is_kota:
        return np.random.uniform(0.60, 0.80)
    elif kab_name in rural:
        return np.random.uniform(0.20, 0.40)
    else:
        return np.random.uniform(0.35, 0.55)

# Generate demographic features for each kecamatan
urbanization = []
populasi_list = []
kepadatan_list = []
income_list = []

for _, row in kecamatan_df.iterrows():
    urb = get_urbanization_factor(row['kabupaten_kota'], row['is_kota'])
    urbanization.append(urb)
    
    # Populasi: log-normal, scaled by urbanization
    # Urban areas: median ~100k-300k, Rural: median ~15k-50k
    base_pop = np.exp(np.random.normal(10.5 + urb * 2.0, 0.6))
    pop = np.clip(base_pop, 5000, 500000)
    populasi_list.append(int(pop))
    
    # Kepadatan: people per km2, estimated from urbanization
    # Urban kecamatan ~5-15 km2, rural ~20-100 km2
    est_area = np.random.uniform(3, 15) if urb > 0.6 else np.random.uniform(15, 80)
    kepadatan = pop / est_area
    kepadatan_list.append(round(kepadatan, 1))
    
    # Income per kapita: juta/bulan, log-normal
    # Higher in developed urban areas
    base_income = np.exp(np.random.normal(1.2 + urb * 1.5, 0.4))
    income = np.clip(base_income, 2.0, 15.0)
    income_list.append(round(income, 2))

kecamatan_df['urbanization_factor'] = urbanization
kecamatan_df['populasi'] = populasi_list
kecamatan_df['kepadatan_penduduk'] = kepadatan_list
kecamatan_df['income_per_kapita'] = income_list

print("Demographic features generated:")
print(f"  Populasi range: {kecamatan_df['populasi'].min():,} - {kecamatan_df['populasi'].max():,}")
print(f"  Kepadatan range: {kecamatan_df['kepadatan_penduduk'].min():.0f} - {kecamatan_df['kepadatan_penduduk'].max():.0f} per km2")
print(f"  Income range: {kecamatan_df['income_per_kapita'].min():.2f} - {kecamatan_df['income_per_kapita'].max():.2f} juta/bulan")
print(f"\nMean by kota/kabupaten type:")
print(kecamatan_df.groupby('is_kota')[['populasi', 'kepadatan_penduduk', 'income_per_kapita']].mean())

Demographic features generated:
  Populasi range: 14,063 - 500,000
  Kepadatan range: 287 - 130842 per km2
  Income range: 2.00 - 15.00 juta/bulan

Mean by kota/kabupaten type:
              populasi  kepadatan_penduduk  income_per_kapita
is_kota                                                      
False    110244.082677         6195.579724           6.929173
True     199103.090909        21356.610227          10.996250


## Section 3: Infrastructure Features

Infrastructure features are driven by urbanization level:
- **jarak_ke_jalan_utama**: Distance to main road (km). Lower in urban areas.
- **jarak_ke_pasar**: Distance to nearest market (km).
- **akses_internet_pct**: Internet access percentage. Higher in urban.
- **skor_infrastruktur**: Composite score (20-95) from weighted average of access metrics.

In [6]:
# Generate infrastructure features based on urbanization
jarak_jalan_list = []
jarak_pasar_list = []
akses_internet_list = []
skor_infra_list = []

for _, row in kecamatan_df.iterrows():
    urb = row['urbanization_factor']
    
    # Distance to main road: lower in urban (0.1-2km), higher in rural (2-15km)
    jarak_jalan = np.random.exponential(1.0 + (1 - urb) * 8.0)
    jarak_jalan = np.clip(jarak_jalan, 0.1, 15.0)
    jarak_jalan_list.append(round(jarak_jalan, 2))
    
    # Distance to market: lower in urban (0.2-3km), higher in rural (3-20km)
    jarak_pasar = np.random.exponential(1.5 + (1 - urb) * 10.0)
    jarak_pasar = np.clip(jarak_pasar, 0.2, 20.0)
    jarak_pasar_list.append(round(jarak_pasar, 2))
    
    # Internet access: higher in urban
    akses_base = 30 + urb * 65 + np.random.normal(0, 8)
    akses_internet = np.clip(akses_base, 30, 95)
    akses_internet_list.append(round(akses_internet, 1))
    
    # Infrastructure composite score
    # Weighted: road access (30%), market access (25%), internet (45%)
    road_score = max(0, 100 - jarak_jalan * 8)  # closer = better
    market_score = max(0, 100 - jarak_pasar * 5)
    internet_score = akses_internet
    
    skor = 0.30 * road_score + 0.25 * market_score + 0.45 * internet_score
    skor = np.clip(skor, 20, 95)
    skor_infra_list.append(round(skor, 1))

kecamatan_df['jarak_ke_jalan_utama'] = jarak_jalan_list
kecamatan_df['jarak_ke_pasar'] = jarak_pasar_list
kecamatan_df['akses_internet_pct'] = akses_internet_list
kecamatan_df['skor_infrastruktur'] = skor_infra_list

print("Infrastructure features generated:")
print(f"  Jarak ke jalan utama: {kecamatan_df['jarak_ke_jalan_utama'].min():.1f} - {kecamatan_df['jarak_ke_jalan_utama'].max():.1f} km")
print(f"  Jarak ke pasar: {kecamatan_df['jarak_ke_pasar'].min():.1f} - {kecamatan_df['jarak_ke_pasar'].max():.1f} km")
print(f"  Akses internet: {kecamatan_df['akses_internet_pct'].min():.0f}% - {kecamatan_df['akses_internet_pct'].max():.0f}%")
print(f"  Skor infrastruktur: {kecamatan_df['skor_infrastruktur'].min():.1f} - {kecamatan_df['skor_infrastruktur'].max():.1f}")

Infrastructure features generated:
  Jarak ke jalan utama: 0.1 - 15.0 km
  Jarak ke pasar: 0.2 - 20.0 km
  Akses internet: 30% - 95%
  Skor infrastruktur: 20.0 - 95.0


## Section 4: Competition Features

Competition is driven by population density - higher density areas have more businesses competing
for the same customer base.

- **jumlah_kompetitor_radius_3km**: Number of competitors within 3km radius (0-50).

In [7]:
# Competition: proportional to population density
kompetitor_list = []

for _, row in kecamatan_df.iterrows():
    # More competitors in denser areas
    density_factor = row['kepadatan_penduduk'] / kecamatan_df['kepadatan_penduduk'].max()
    base_kompetitor = density_factor * 40 + np.random.normal(0, 5)
    kompetitor = int(np.clip(base_kompetitor, 0, 50))
    kompetitor_list.append(kompetitor)

kecamatan_df['jumlah_kompetitor_radius_3km'] = kompetitor_list

print("Competition features generated:")
print(f"  Kompetitor range: {kecamatan_df['jumlah_kompetitor_radius_3km'].min()} - {kecamatan_df['jumlah_kompetitor_radius_3km'].max()}")
print(f"  Mean: {kecamatan_df['jumlah_kompetitor_radius_3km'].mean():.1f}")
print(f"\nCorrelation with kepadatan_penduduk: {kecamatan_df['jumlah_kompetitor_radius_3km'].corr(kecamatan_df['kepadatan_penduduk']):.3f}")

Competition features generated:
  Kompetitor range: 0 - 40
  Mean: 3.5



Correlation with kepadatan_penduduk: 0.750


## Section 5: Financial Access Features

Financial infrastructure availability is driven by urbanization:
- **jarak_ke_bank_terdekat**: Distance to nearest bank (km). Lower in urban areas.
- **penetrasi_kur_pct**: KUR (Kredit Usaha Rakyat) government program penetration rate (5-45%).

In [8]:
# Financial access features
jarak_bank_list = []
penetrasi_kur_list = []

for _, row in kecamatan_df.iterrows():
    urb = row['urbanization_factor']
    
    # Distance to bank: urban 0.1-3km, rural 3-25km
    jarak_bank = np.random.exponential(1.0 + (1 - urb) * 12.0)
    jarak_bank = np.clip(jarak_bank, 0.1, 25.0)
    jarak_bank_list.append(round(jarak_bank, 2))
    
    # KUR penetration: varies 5-45%, slightly higher in semi-urban areas
    # (rural too remote for program reach, ultra-urban less need)
    kur_base = 15 + urb * 20 + np.random.normal(0, 7)
    # Slightly boost mid-range urbanization
    if 0.4 < urb < 0.7:
        kur_base += 5
    kur = np.clip(kur_base, 5, 45)
    penetrasi_kur_list.append(round(kur, 1))

kecamatan_df['jarak_ke_bank_terdekat'] = jarak_bank_list
kecamatan_df['penetrasi_kur_pct'] = penetrasi_kur_list

print("Financial access features generated:")
print(f"  Jarak ke bank: {kecamatan_df['jarak_ke_bank_terdekat'].min():.1f} - {kecamatan_df['jarak_ke_bank_terdekat'].max():.1f} km")
print(f"  Penetrasi KUR: {kecamatan_df['penetrasi_kur_pct'].min():.1f}% - {kecamatan_df['penetrasi_kur_pct'].max():.1f}%")

Financial access features generated:
  Jarak ke bank: 0.1 - 25.0 km
  Penetrasi KUR: 5.0% - 45.0%


## Section 6: Risk Features (from Actual Geography)

Risk features are based on actual geographic hazard zones in Jawa Barat:

- **risiko_banjir** (0-1): HIGH for northern coastal areas (Indramayu, Karawang, Kab. Bekasi, northern Subang) 
  due to flat terrain and river flooding. LOW for mountainous southern areas.
- **risiko_gempa** (0-1): HIGH for southern areas along the subduction zone (Garut, Tasikmalaya, 
  Pangandaran, Ciamis, southern Sukabumi). LOW for northern plains.

These are based on actual BNPB (national disaster agency) hazard maps.

In [9]:
# Risk features based on actual geographic hazard zones
def get_flood_risk(kab_name, lat):
    """Flood risk based on location. Northern coastal = high risk."""
    # High flood risk: northern coastal lowlands
    if kab_name == 'Kab. Indramayu':
        return np.random.uniform(0.6, 0.9)
    elif kab_name == 'Kab. Karawang':
        return np.random.uniform(0.5, 0.8)
    elif kab_name in ['Kab. Bekasi', 'Kota Bekasi']:
        return np.random.uniform(0.4, 0.7)
    elif kab_name == 'Kab. Subang' and lat > -6.5:
        return np.random.uniform(0.4, 0.6)
    elif kab_name in ['Kab. Cirebon', 'Kota Cirebon']:
        return np.random.uniform(0.3, 0.6)
    elif kab_name == 'Kab. Subang':
        return np.random.uniform(0.2, 0.4)
    elif lat > -6.5:  # General northern areas
        return np.random.uniform(0.2, 0.4)
    elif lat < -7.0:  # Southern mountain areas - low flood
        return np.random.uniform(0.05, 0.2)
    else:
        return np.random.uniform(0.1, 0.3)

def get_earthquake_risk(kab_name, lat):
    """Earthquake risk based on location. Southern = high risk (near subduction zone)."""
    # High earthquake risk: southern areas near plate boundary
    if kab_name == 'Kab. Pangandaran':
        return np.random.uniform(0.6, 0.9)
    elif kab_name in ['Kab. Garut', 'Kab. Tasikmalaya', 'Kota Tasikmalaya']:
        return np.random.uniform(0.5, 0.8)
    elif kab_name == 'Kab. Ciamis':
        return np.random.uniform(0.4, 0.7)
    elif kab_name in ['Kab. Sukabumi', 'Kota Sukabumi']:
        return np.random.uniform(0.5, 0.7)
    elif kab_name == 'Kab. Bandung' and lat < -7.0:
        return np.random.uniform(0.3, 0.5)
    elif lat < -7.0:  # General southern
        return np.random.uniform(0.3, 0.6)
    elif lat > -6.5:  # Northern plains - low earthquake
        return np.random.uniform(0.05, 0.2)
    else:  # Central band
        return np.random.uniform(0.15, 0.35)

risiko_banjir_list = []
risiko_gempa_list = []

for _, row in kecamatan_df.iterrows():
    banjir = get_flood_risk(row['kabupaten_kota'], row['latitude'])
    gempa = get_earthquake_risk(row['kabupaten_kota'], row['latitude'])
    risiko_banjir_list.append(round(banjir, 3))
    risiko_gempa_list.append(round(gempa, 3))

kecamatan_df['risiko_banjir'] = risiko_banjir_list
kecamatan_df['risiko_gempa'] = risiko_gempa_list

print("Risk features generated:")
print(f"  Risiko banjir: {kecamatan_df['risiko_banjir'].min():.3f} - {kecamatan_df['risiko_banjir'].max():.3f}")
print(f"  Risiko gempa: {kecamatan_df['risiko_gempa'].min():.3f} - {kecamatan_df['risiko_gempa'].max():.3f}")
print(f"\nTop flood risk areas (mean):")
flood_by_kab = kecamatan_df.groupby('kabupaten_kota')['risiko_banjir'].mean().sort_values(ascending=False)
print(flood_by_kab.head(5).to_string())
print(f"\nTop earthquake risk areas (mean):")
quake_by_kab = kecamatan_df.groupby('kabupaten_kota')['risiko_gempa'].mean().sort_values(ascending=False)
print(quake_by_kab.head(5).to_string())

Risk features generated:
  Risiko banjir: 0.050 - 0.883
  Risiko gempa: 0.051 - 0.885

Top flood risk areas (mean):
kabupaten_kota
Kab. Indramayu    0.760968
Kab. Karawang     0.624267
Kab. Bekasi       0.565304
Kota Bekasi       0.507583
Kab. Cirebon      0.464300

Top earthquake risk areas (mean):
kabupaten_kota
Kab. Pangandaran    0.718000
Kab. Tasikmalaya    0.657179
Kab. Garut          0.654881
Kota Tasikmalaya    0.628200
Kota Sukabumi       0.619429


## Section 7: Business Features (10,000 UMKMs)

We generate 10,000 UMKM records distributed across kecamatan proportional to population.
Each UMKM has:
- **jenis_usaha**: Business type (Makanan 40%, Fashion 20%, Kerajinan 15%, Jasa 15%, Pertanian 10%)
- **tahun_berdiri**: Year established (2010-2024, uniform)
- **jumlah_karyawan**: Employee count (1-50, right-skewed, median 3-5)
- **has_digital_presence**: Digital presence (higher in urban 60-80%, lower in rural 15-35%)
- **omset_bulanan**: Monthly revenue in juta (based on business type, location, and age -- NOT from score)

In [10]:
# Distribute 10,000 UMKMs across kecamatan proportional to population
N_UMKM = 10000

# Calculate allocation proportional to population
pop_total = kecamatan_df['populasi'].sum()
kecamatan_df['umkm_allocation'] = (kecamatan_df['populasi'] / pop_total * N_UMKM).round().astype(int)

# Adjust to hit exactly 10,000
diff = N_UMKM - kecamatan_df['umkm_allocation'].sum()
if diff != 0:
    # Add/subtract from largest kecamatan
    idx_largest = kecamatan_df['populasi'].nlargest(abs(diff)).index
    kecamatan_df.loc[idx_largest, 'umkm_allocation'] += np.sign(diff)

print(f"Total UMKM to generate: {kecamatan_df['umkm_allocation'].sum()}")
print(f"Kecamatan with most UMKMs: {kecamatan_df.loc[kecamatan_df['umkm_allocation'].idxmax(), 'kecamatan']} "
      f"({kecamatan_df['umkm_allocation'].max()} UMKMs)")
print(f"Kecamatan with fewest UMKMs: {kecamatan_df['umkm_allocation'].min()}")

Total UMKM to generate: 10000
Kecamatan with most UMKMs: Rancaekek (69 UMKMs)
Kecamatan with fewest UMKMs: 2


In [11]:
# Generate individual UMKM records
jenis_usaha_choices = ['Makanan', 'Fashion', 'Kerajinan', 'Jasa', 'Pertanian']
jenis_usaha_probs = [0.40, 0.20, 0.15, 0.15, 0.10]

# Base monthly revenue by business type (in juta rupiah)
omset_base = {
    'Makanan': 30,
    'Fashion': 45,
    'Kerajinan': 25,
    'Jasa': 50,
    'Pertanian': 20
}

umkm_records = []

for _, kec_row in kecamatan_df.iterrows():
    n_umkm = int(kec_row['umkm_allocation'])
    if n_umkm <= 0:
        continue
    
    for _ in range(n_umkm):
        # Business type
        jenis = np.random.choice(jenis_usaha_choices, p=jenis_usaha_probs)
        
        # Year established
        tahun = np.random.randint(2010, 2025)
        
        # Business age
        business_age = 2024 - tahun
        
        # Employees: right-skewed (log-normal, median 3-5)
        karyawan = int(np.clip(np.random.lognormal(1.2, 0.7), 1, 50))
        
        # Digital presence: higher in urban areas
        urb = kec_row['urbanization_factor']
        digital_prob = 0.15 + urb * 0.65  # 15% rural to 80% urban
        has_digital = int(np.random.random() < digital_prob)
        
        # Monthly revenue (omset_bulanan) in juta
        # Based on: business type base + location quality + age factor + noise
        base = omset_base[jenis]
        location_mult = 0.6 + urb * 0.8  # 0.6x rural to 1.4x urban
        age_mult = 1.0 + business_age * 0.05  # older businesses earn more
        digital_mult = 1.2 if has_digital else 1.0
        noise = np.random.lognormal(0, 0.4)
        
        omset = base * location_mult * age_mult * digital_mult * noise
        omset = np.clip(omset, 5, 500)
        
        umkm_records.append({
            'kecamatan': kec_row['kecamatan'],
            'kabupaten_kota': kec_row['kabupaten_kota'],
            'latitude': kec_row['latitude'] + np.random.uniform(-0.01, 0.01),
            'longitude': kec_row['longitude'] + np.random.uniform(-0.01, 0.01),
            'is_kota': kec_row['is_kota'],
            'jenis_usaha': jenis,
            'tahun_berdiri': tahun,
            'jumlah_karyawan': karyawan,
            'has_digital_presence': has_digital,
            'omset_bulanan': round(omset, 2),
            # Carry forward kecamatan-level features
            'populasi': kec_row['populasi'],
            'kepadatan_penduduk': kec_row['kepadatan_penduduk'],
            'income_per_kapita': kec_row['income_per_kapita'],
            'jarak_ke_jalan_utama': kec_row['jarak_ke_jalan_utama'],
            'jarak_ke_pasar': kec_row['jarak_ke_pasar'],
            'akses_internet_pct': kec_row['akses_internet_pct'],
            'skor_infrastruktur': kec_row['skor_infrastruktur'],
            'jumlah_kompetitor_radius_3km': kec_row['jumlah_kompetitor_radius_3km'],
            'jarak_ke_bank_terdekat': kec_row['jarak_ke_bank_terdekat'],
            'penetrasi_kur_pct': kec_row['penetrasi_kur_pct'],
            'risiko_banjir': kec_row['risiko_banjir'],
            'risiko_gempa': kec_row['risiko_gempa'],
        })

umkm_df = pd.DataFrame(umkm_records)
print(f"Generated {len(umkm_df)} UMKM records")
print(f"\nBusiness type distribution:")
print(umkm_df['jenis_usaha'].value_counts(normalize=True).to_string())
print(f"\nEmployee stats: median={umkm_df['jumlah_karyawan'].median()}, mean={umkm_df['jumlah_karyawan'].mean():.1f}")
print(f"Digital presence: {umkm_df['has_digital_presence'].mean()*100:.1f}%")
print(f"Omset range: {umkm_df['omset_bulanan'].min():.1f} - {umkm_df['omset_bulanan'].max():.1f} juta/bulan")

Generated 10000 UMKM records

Business type distribution:
jenis_usaha
Makanan      0.3971
Fashion      0.2026
Kerajinan    0.1515
Jasa         0.1495
Pertanian    0.0993

Employee stats: median=3.0, mean=3.7
Digital presence: 51.2%
Omset range: 7.9 - 486.6 juta/bulan


## Section 8: Target Variable Generation (NO Circular Logic)

The target variable `is_survived_3yr` is generated using a **known logistic relationship** from features.
This ensures:
1. Features CAUSE the target (not the other way around)
2. The relationship is known, so we can validate model recovery
3. No information leakage

**Logistic model:**
```
logit = 0.3*norm(skor_infrastruktur) + 0.2*norm(income_per_kapita) 
        - 0.25*norm(jumlah_kompetitor) - 0.2*risiko_banjir - 0.15*risiko_gempa 
        + 0.15*has_digital_presence + 0.1*norm(business_age) 
        + 0.1*norm(omset_bulanan) - 0.1*norm(jarak_ke_bank) + noise
```

Target survival rate: ~65-70%

In [12]:
# Helper function to normalize a series to [0, 1]
def normalize(series):
    """Min-max normalize to [0, 1]."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Compute business age
umkm_df['business_age'] = 2024 - umkm_df['tahun_berdiri']

# Normalize features for logistic model
norm_infra = normalize(umkm_df['skor_infrastruktur'])
norm_income = normalize(umkm_df['income_per_kapita'])
norm_kompetitor = normalize(umkm_df['jumlah_kompetitor_radius_3km'])
norm_age = normalize(umkm_df['business_age'])
norm_omset = normalize(umkm_df['omset_bulanan'])
norm_jarak_bank = normalize(umkm_df['jarak_ke_bank_terdekat'])

# Generate logit with known coefficients
logit = (
    0.30 * norm_infra
    + 0.20 * norm_income
    - 0.25 * norm_kompetitor
    - 0.20 * umkm_df['risiko_banjir']
    - 0.15 * umkm_df['risiko_gempa']
    + 0.15 * umkm_df['has_digital_presence']
    + 0.10 * norm_age
    + 0.10 * norm_omset
    - 0.10 * norm_jarak_bank
)

# Add noise and shift to achieve ~65-70% survival
noise = np.random.normal(0, 0.15, len(umkm_df))
logit_with_noise = logit + noise

# Shift the logit to target ~67% survival rate
# First check current mean probability
raw_prob = 1 / (1 + np.exp(-logit_with_noise))
current_mean = raw_prob.mean()
print(f"Raw mean survival probability: {current_mean:.3f}")

# Adjust intercept to target ~0.67
target_rate = 0.67
# Use bisection to find the right shift
shift_low, shift_high = -5.0, 5.0
for _ in range(50):
    shift_mid = (shift_low + shift_high) / 2
    test_prob = 1 / (1 + np.exp(-(logit_with_noise + shift_mid)))
    if test_prob.mean() < target_rate:
        shift_low = shift_mid
    else:
        shift_high = shift_mid

optimal_shift = (shift_low + shift_high) / 2
print(f"Optimal intercept shift: {optimal_shift:.4f}")

# Apply the shift and generate probabilities
prob_survival = 1 / (1 + np.exp(-(logit_with_noise + optimal_shift)))
umkm_df['prob_survival'] = prob_survival

# Generate binary target from Bernoulli
umkm_df['is_survived_3yr'] = (np.random.random(len(umkm_df)) < prob_survival).astype(int)

actual_rate = umkm_df['is_survived_3yr'].mean()
print(f"\nActual survival rate: {actual_rate:.3f} ({actual_rate*100:.1f}%)")
print(f"Probability distribution:")
print(f"  Mean: {prob_survival.mean():.3f}")
print(f"  Std:  {prob_survival.std():.3f}")
print(f"  Min:  {prob_survival.min():.3f}")
print(f"  Max:  {prob_survival.max():.3f}")

Raw mean survival probability: 0.567
Optimal intercept shift: 0.4450

Actual survival rate: 0.680 (68.0%)
Probability distribution:
  Mean: 0.670
  Std:  0.046
  Min:  0.506
  Max:  0.809


## Section 9: Location Potential Score (skor_potensi)

`skor_potensi` is a **DERIVED** metric - computed from features, NOT randomly generated.
This is an interpretable composite score representing the overall business potential of a location.

**Formula:**
```
skor_potensi = 0.25 * norm(skor_infrastruktur) 
             + 0.20 * norm(income_per_kapita) 
             + 0.15 * norm(akses_internet)
             + 0.15 * (1 - norm(jumlah_kompetitor/max))
             + 0.10 * (1 - risiko_composite)
             + 0.10 * norm(penetrasi_kur)
             + 0.05 * has_digital_presence
```
Scaled to 0-100.

In [13]:
# Compute skor_potensi as a DERIVED metric from features
# All normalization is min-max to [0, 1]
norm_infra_score = normalize(umkm_df['skor_infrastruktur'])
norm_income_score = normalize(umkm_df['income_per_kapita'])
norm_internet = normalize(umkm_df['akses_internet_pct'])
norm_kompetitor_inv = 1 - normalize(umkm_df['jumlah_kompetitor_radius_3km'])
risiko_composite = (umkm_df['risiko_banjir'] + umkm_df['risiko_gempa']) / 2
norm_risk_inv = 1 - risiko_composite
norm_kur = normalize(umkm_df['penetrasi_kur_pct'])

# Weighted combination
skor_raw = (
    0.25 * norm_infra_score
    + 0.20 * norm_income_score
    + 0.15 * norm_internet
    + 0.15 * norm_kompetitor_inv
    + 0.10 * norm_risk_inv
    + 0.10 * norm_kur
    + 0.05 * umkm_df['has_digital_presence']
)

# Scale to 0-100
skor_min = skor_raw.min()
skor_max = skor_raw.max()
umkm_df['skor_potensi'] = ((skor_raw - skor_min) / (skor_max - skor_min) * 100).round(2)

print("skor_potensi (Location Potential Score) generated:")
print(f"  Range: {umkm_df['skor_potensi'].min():.1f} - {umkm_df['skor_potensi'].max():.1f}")
print(f"  Mean: {umkm_df['skor_potensi'].mean():.1f}")
print(f"  Std:  {umkm_df['skor_potensi'].std():.1f}")
print(f"\nCorrelation of skor_potensi with survival:")
print(f"  {umkm_df['skor_potensi'].corr(umkm_df['is_survived_3yr']):.3f}")
print(f"\nThis confirms the score is meaningful - higher score correlates with survival,")
print(f"because both are driven by the same underlying features (no circular logic).")

skor_potensi (Location Potential Score) generated:
  Range: 0.0 - 100.0
  Mean: 56.2
  Std:  19.7

Correlation of skor_potensi with survival:
  0.057

This confirms the score is meaningful - higher score correlates with survival,
because both are driven by the same underlying features (no circular logic).


## Section 10: Save Outputs & Data Quality Checks

Save the generated data:
1. `data/kecamatan_reference.csv` - Kecamatan reference with geography and demographics
2. `data/umkm_dataset.csv` - Full UMKM dataset (10,000 records)

Then perform data quality validation.

In [14]:
# Save kecamatan reference data
kec_save_cols = ['kabupaten_kota', 'kecamatan', 'latitude', 'longitude', 'is_kota',
                 'populasi', 'kepadatan_penduduk', 'income_per_kapita',
                 'jarak_ke_jalan_utama', 'jarak_ke_pasar', 'akses_internet_pct',
                 'skor_infrastruktur', 'jumlah_kompetitor_radius_3km',
                 'jarak_ke_bank_terdekat', 'penetrasi_kur_pct',
                 'risiko_banjir', 'risiko_gempa']

kecamatan_df[kec_save_cols].to_csv('../data/kecamatan_reference.csv', index=False)
print(f"Saved kecamatan reference: ../data/kecamatan_reference.csv ({len(kecamatan_df)} rows)")

# Save UMKM dataset
umkm_save_cols = ['kabupaten_kota', 'kecamatan', 'latitude', 'longitude', 'is_kota',
                  'jenis_usaha', 'tahun_berdiri', 'jumlah_karyawan',
                  'has_digital_presence', 'omset_bulanan',
                  'populasi', 'kepadatan_penduduk', 'income_per_kapita',
                  'jarak_ke_jalan_utama', 'jarak_ke_pasar', 'akses_internet_pct',
                  'skor_infrastruktur', 'jumlah_kompetitor_radius_3km',
                  'jarak_ke_bank_terdekat', 'penetrasi_kur_pct',
                  'risiko_banjir', 'risiko_gempa',
                  'skor_potensi', 'is_survived_3yr']

umkm_df[umkm_save_cols].to_csv('../data/umkm_dataset.csv', index=False)
print(f"Saved UMKM dataset: ../data/umkm_dataset.csv ({len(umkm_df)} rows, {len(umkm_save_cols)} columns)")

Saved kecamatan reference: ../data/kecamatan_reference.csv (596 rows)
Saved UMKM dataset: ../data/umkm_dataset.csv (10000 rows, 24 columns)


In [15]:
# Data quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

df = pd.read_csv('../data/umkm_dataset.csv')
kec = pd.read_csv('../data/kecamatan_reference.csv')

# 1. Row counts
print(f"\n1. Row counts:")
print(f"   UMKM dataset: {len(df)} rows (expected: 10,000)")
print(f"   Kecamatan reference: {len(kec)} rows (expected: ~627)")
assert len(df) == 10000, f"Expected 10000 rows, got {len(df)}"
assert len(kec) >= 500, f"Expected >=500 kecamatan, got {len(kec)}"

# 2. No null values
print(f"\n2. Null values:")
print(f"   UMKM nulls: {df.isnull().sum().sum()}")
print(f"   Kecamatan nulls: {kec.isnull().sum().sum()}")
assert df.isnull().sum().sum() == 0, "Found null values in UMKM dataset"

# 3. Geographic bounds (Jawa Barat)
print(f"\n3. Geographic bounds:")
print(f"   Lat range: {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
print(f"   Lon range: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")
assert df['latitude'].min() > -8.0, "Latitude too far south"
assert df['latitude'].max() < -6.0, "Latitude too far north"
assert df['longitude'].min() > 106.0, "Longitude too far west"
assert df['longitude'].max() < 109.0, "Longitude too far east"

# 4. Target distribution
survival_rate = df['is_survived_3yr'].mean()
print(f"\n4. Target distribution:")
print(f"   Survival rate: {survival_rate:.3f} ({survival_rate*100:.1f}%)")
assert 0.55 < survival_rate < 0.80, f"Survival rate {survival_rate} outside expected range"

# 5. Score distribution
print(f"\n5. skor_potensi distribution:")
print(f"   Range: {df['skor_potensi'].min():.1f} - {df['skor_potensi'].max():.1f}")
print(f"   Mean: {df['skor_potensi'].mean():.1f}, Std: {df['skor_potensi'].std():.1f}")

# 6. Check no circular logic - correlation matrix
print(f"\n6. Key correlations (verifying no circular logic):")
key_cols = ['skor_potensi', 'omset_bulanan', 'is_survived_3yr', 'skor_infrastruktur']
corr = df[key_cols].corr()
print(corr.to_string())

# 7. Verify risk features match geography
print(f"\n7. Risk validation:")
north_flood = df[df['kabupaten_kota'] == 'Kab. Indramayu']['risiko_banjir'].mean()
south_quake = df[df['kabupaten_kota'] == 'Kab. Pangandaran']['risiko_gempa'].mean()
print(f"   Indramayu flood risk (should be high): {north_flood:.3f}")
print(f"   Pangandaran earthquake risk (should be high): {south_quake:.3f}")
assert north_flood > 0.5, "Indramayu should have high flood risk"
assert south_quake > 0.5, "Pangandaran should have high earthquake risk"

print(f"\n{'=' * 60}")
print("ALL DATA QUALITY CHECKS PASSED!")
print(f"{'=' * 60}")

DATA QUALITY CHECKS



1. Row counts:
   UMKM dataset: 10000 rows (expected: 10,000)
   Kecamatan reference: 596 rows (expected: ~627)

2. Null values:
   UMKM nulls: 0
   Kecamatan nulls: 0

3. Geographic bounds:
   Lat range: -7.7569 to -6.1550
   Lon range: 106.7041 to 108.7192

4. Target distribution:
   Survival rate: 0.680 (68.0%)

5. skor_potensi distribution:
   Range: 0.0 - 100.0
   Mean: 56.2, Std: 19.7

6. Key correlations (verifying no circular logic):
                    skor_potensi  omset_bulanan  is_survived_3yr  skor_infrastruktur
skor_potensi            1.000000       0.266055         0.056571            0.803317
omset_bulanan           0.266055       1.000000         0.012721            0.196853
is_survived_3yr         0.056571       0.012721         1.000000            0.039412
skor_infrastruktur      0.803317       0.196853         0.039412            1.000000

7. Risk validation:
   Indramayu flood risk (should be high): 0.765
   Pangandaran earthquake risk (should be high): 0.726

ALL

## Summary

This notebook generated:
1. **Kecamatan Reference** (~627 entries): Real kecamatan names across 27 kabupaten/kota in Jawa Barat with geographic coordinates, demographic, and infrastructure features.
2. **UMKM Dataset** (10,000 records): Realistic business data with features generated from geography and demographics.

**Key Design Guarantees:**
- All features are generated independently from geographic/demographic basis
- `is_survived_3yr` is generated FROM features using a known logistic model
- `skor_potensi` is COMPUTED from features (weighted formula)
- No circular logic exists in the data generation pipeline

The data is ready for use in subsequent notebooks:
- Notebook 02: Exploratory Data Analysis
- Notebook 03: Clustering & Segmentation
- Notebook 04: Predictive Modeling
- Notebook 05: Recommendation System